# Import

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import wandb
import random

from typing import List
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from transformers import DataCollatorForTokenClassification

# Settings

In [31]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [32]:
set_seed(1)

In [33]:
PERSONAL_PATH = r"/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 1"

FOLDER_PATH = os.path.join(PERSONAL_PATH, "data")
OUTPUT_PATH = os.path.join(PERSONAL_PATH, "models")
OUTPUT_FILES_PATH = os.path.join(PERSONAL_PATH,"output")
TRAIN_PATH = os.path.join(FOLDER_PATH, "manzoni_train_tokens.csv")
DEV_PATH = os.path.join(FOLDER_PATH, "manzoni_dev_tokens.csv")
TEST_PATH = os.path.join(FOLDER_PATH, "OOD_test.csv")

SEQ_LEN   = 64
BATCH_SIZE = 16
EPOCHS     = 20
LR         = 2e-5
DROPOUT    = 0.2
FREEZE_N   = 4
LOG_EVERY  = 10
MODEL_NAME  = "dbmdz/bert-base-italian-cased"
RUN_NAME    = "it-bert"
PROJECT     = "sentence-splitting_2"

# Utils

In [34]:
def read_tokens(path: str):
    """Return the *flat* list of tokens and their EOS labels."""
    df = pd.read_csv(path)
    return list(df.token.astype(str)), list(df.label.astype(int))

In [35]:
def chunkify(tokens, labels, size: int):
    """Split a flat sequence into fixed‑length chunks, padding the last chunk.

    Padding tokens receive label ‑100 so they are ignored by the loss &
    metrics.
    """
    assert len(tokens) == len(labels)
    chunks_toks, chunks_labs = [], []
    pad_tok, pad_lab = "[PAD]", -100
    for start in range(0, len(tokens), size):
        t = tokens[start : start + size]
        l = labels[start : start + size]
        if len(t) < size:
            pad_len = size - len(t)
            t += [pad_tok] * pad_len
            l += [pad_lab] * pad_len
        chunks_toks.append(t)
        chunks_labs.append(l)
    return chunks_toks, chunks_labs

In [36]:
def load_data() -> DatasetDict:
    """Read csv files → fixed‑length chunks → HuggingFace `DatasetDict`."""
    tr_tok, tr_lab = read_tokens(TRAIN_PATH)
    dv_tok, dv_lab = read_tokens(DEV_PATH)

    tr_toks, tr_labs = chunkify(tr_tok, tr_lab, SEQ_LEN)
    dv_toks, dv_labs = chunkify(dv_tok, dv_lab, SEQ_LEN)

    return DatasetDict(
        {
            "train": Dataset.from_dict({"tokens": tr_toks, "labels": tr_labs}),
            "dev":   Dataset.from_dict({"tokens": dv_toks, "labels": dv_labs}),
        }
    )

In [37]:
def tokenize(batch):
    enc = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        padding=False,
        truncation=False
    )
    aligned_labels = []
    for i, word_ids in enumerate(enc.word_ids(batch_index=i) for i in range(len(batch["tokens"]))):
        labs, mask, prev = batch["labels"][i], [], None
        for w in word_ids:
            if w is None:
                mask.append(-100)
            elif w != prev:
                mask.append(labs[w])
                prev = w
            else:
                mask.append(-100)
        aligned_labels.append(mask)
    enc["labels"] = aligned_labels
    return enc

In [38]:
def eos_stats(labels_2d):
    """Return EOS ratio and balanced class‑weights."""
    flat = np.concatenate(labels_2d)
    mask = flat != -100
    flat = flat[mask]
    ratio = flat.mean()
    weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=flat)
    return ratio, torch.tensor(weights, dtype=torch.float)

In [39]:
def freeze_layers(model, n):
    '''
    Freeze the first n layers of the model.
    '''
    if n <= 0:
        return
    for p in model.bert.embeddings.parameters():
        p.requires_grad = False
    for layer in model.bert.encoder.layer[:n]:
        for p in layer.parameters():
            p.requires_grad = False

In [40]:
def compute_metrics(p):
    '''
    Compute metrics for the model.
    '''
    preds  = p.predictions.argmax(-1).ravel()
    labels = p.label_ids.ravel()
    mask   = labels != -100
    preds, labels = preds[mask], labels[mask]

    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=[1], average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision_EOS": prec,
        "recall_EOS":    rec,
        "f1_EOS":        f1,
    }

In [41]:
class WeightedTrainer(Trainer):
    """Trainer with class‑weighted CE loss (ignores sub‑tokens)."""

    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = CrossEntropyLoss(
            weight=self.class_weights.to(logits.device), ignore_index=-100
        )
        loss = loss_fn(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [42]:
def split_with_model(
    tokens: List[str],
    model: AutoModelForTokenClassification,
    tok: AutoTokenizer,
    seq_len: int = SEQ_LEN,
) -> List[str]:
    """
    Model inference on all tokens. The tokens in the set are seen as one big text,
    so this function runs a window of length SEQ_LEN over the text, inference occurs,
    then the sentence is split when an EOS is found,
    and the remaining tokens in the sequence are used as leftovers for the next sequence.
    If the entire sequence is predicted as non-EOS, it is saved in a baffer,
    and only downloaded when an EOS is predicted.
    """
    device = next(model.parameters()).device
    model.eval()

    sentences: List[str] = []
    pending:   List[str] = []
    idx = 0

    with torch.no_grad():
        while idx < len(tokens):
            window = tokens[idx: idx + seq_len]

            enc = tok(
                window,
                is_split_into_words=True,
                truncation=True,
                max_length=seq_len,
                padding="max_length",
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits
            preds = logits.argmax(-1)[0].cpu().tolist()

            word_ids   = tok(window, is_split_into_words=True).word_ids()
            word_preds = []
            seen = None
            for p, w in zip(preds, word_ids):
                if w is None or w == seen:
                    continue
                word_preds.append(p)
                seen = w

            if 1 not in word_preds:
                pending.extend(window)
                idx += seq_len
                continue

            for j, (tok_word, pred) in enumerate(zip(window, word_preds)):
                pending.append(tok_word)
                if pred == 1:
                    sentences.append(" ".join(pending))
                    pending = []
                    idx += j + 1
                    break

    if pending:
        sentences.append(" ".join(pending))

    return sentences

# Main

In [43]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForTokenClassification(tokenizer)
wandb.init(project=PROJECT, name=RUN_NAME)

eval/accuracy,▁▁
eval/f1_EOS,▁▁
eval/loss,▁▁
eval/precision_EOS,▁▁
eval/recall_EOS,▁▁
eval/runtime,▁█
eval/samples_per_second,█▁
eval/steps_per_second,█▁
test/accuracy,▁▁
test/f1_EOS,▁▁
+8,...


In [44]:
# 1. Load data
ds = load_data()
ratio, cls_w = eos_stats(ds["train"]["labels"])
print(f"EOS ratio: {ratio*100:.2f}%  |  class weights: {cls_w.tolist()}")

EOS ratio: 3.27%  |  class weights: [0.5169185400009155, 15.276665687561035]


In [45]:
ds

DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1169
    })
    dev: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 146
    })
})

In [46]:
# 2. Tokenizer and model
tokenised = ds.map(tokenize, batched=True, remove_columns=["tokens", "labels"])

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "NOT_EOS", 1: "EOS"},
    label2id={"NOT_EOS": 0, "EOS": 1},
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT,
)
freeze_layers(model, FREEZE_N)

Map:   0%|          | 0/1169 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dbmdz/bert-base-italian-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [47]:
# 3. Training setting
args = TrainingArguments(
    output_dir="checkpoints",
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=LOG_EVERY,
    run_name=RUN_NAME,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True,
    report_to=["wandb"],
    seed=1
)

In [48]:
trainer = WeightedTrainer(
    cls_w,
    model=model,
    args=args,
    train_dataset=tokenised["train"],
    eval_dataset=tokenised["dev"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

/tmp/ipython-input-1059353679.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [49]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Eos,Recall Eos,F1 Eos
1,0.037300,0.008391,0.997324,0.928367,1.000000,0.962853
2,0.005300,0.006810,0.998609,0.964179,0.996914,0.980273
3,0.005400,0.004469,0.998716,0.967066,0.996914,0.981763
4,0.019700,0.004484,0.998502,0.961310,0.996914,0.978788
5,0.010500,0.007651,0.999037,0.978723,0.993827,0.986217
6,0.002800,0.010392,0.998930,0.975758,0.993827,0.984709
7,0.013900,0.007152,0.999144,0.981707,0.993827,0.987730
8,0.008600,0.002846,0.999144,0.975904,1.000000,0.987805
9,0.001400,0.004513,0.999251,0.981763,0.996914,0.989280
10,0.004500,0.004094,0.998823,0.969970,0.996914,0.983257


TrainOutput(global_step=1480, training_loss=0.012019961743420493, metrics={'train_runtime': 132.5464, 'train_samples_per_second': 176.391, 'train_steps_per_second': 11.166, 'total_flos': 1050401734483704.0, 'train_loss': 0.012019961743420493, 'epoch': 20.0})

In [50]:
# 4. Quick Dev‑set evaluation & diagnostics
dev_eval = trainer.evaluate()
print("\nDev metrics:", dev_eval)
a=len(tokenised["dev"])
print(f"len di dev: {a}")
pred_out = trainer.predict(tokenised["dev"])
y_true = pred_out.label_ids.ravel()
y_pred = pred_out.predictions.argmax(-1).ravel()
mask = y_true != -100
y_true, y_pred = y_true[mask], y_pred[mask]

print(
    "\nClassification report (Dev):\n",
    classification_report(y_true, y_pred, target_names=["NOT_EOS", "EOS"]),
)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["NOT_EOS", "EOS"],
    yticklabels=["NOT_EOS", "EOS"],
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix – Dev")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FILES_PATH,"confusion_matrix_dev.png"))
plt.close()
wandb.log({"confusion_matrix": wandb.Image(os.path.join(OUTPUT_FILES_PATH,"confusion_matrix_dev.png"))})


Dev metrics: {'eval_loss': 0.01058201864361763, 'eval_accuracy': 0.9992507759820186, 'eval_precision_EOS': 0.9847094801223242, 'eval_recall_EOS': 0.9938271604938271, 'eval_f1_EOS': 0.989247311827957, 'eval_runtime': 0.2054, 'eval_samples_per_second': 710.903, 'eval_steps_per_second': 24.346, 'epoch': 20.0}
len di dev: 146

Classification report (Dev):
               precision    recall  f1-score   support

     NOT_EOS       1.00      1.00      1.00      9019
         EOS       0.98      0.99      0.99       324

    accuracy                           1.00      9343
   macro avg       0.99      1.00      0.99      9343
weighted avg       1.00      1.00      1.00      9343



In [51]:
# 5. Sliding‑window inference on the *flat* dev tokens
dev_tokens, _ = read_tokens(DEV_PATH)
sentences_pred = split_with_model(dev_tokens, model, tokenizer, SEQ_LEN)
out_path = os.path.join(OUTPUT_FILES_PATH, "dev_pred_sentences.txt")
with open(out_path, "w", encoding="utf-8") as fp:
    fp.write("\n * \n".join(sentences_pred))

print(f"Saved predicted sentences to: {out_path}")
wandb.finish()

Saved predicted sentences to: /content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 1/output/dev_pred_sentences.txt


eval/accuracy,▁▅▆▅▇▆▇▇▇▆▇██████▇▇▇▇
eval/f1_EOS,▁▅▅▅▇▆▇▇▇▆▇██████▇▇▇▇
eval/loss,▅▄▂▂▄▆▄▁▂▂▆▇▇█▅▅▆█▇▇▇
eval/precision_EOS,▁▅▆▅▇▇▇▇▇▆▇██████████
eval/recall_EOS,█▄▄▄▁▁▁█▄▄▄▄▄▄▄▄▄▁▁▁▁
eval/runtime,▁▁▃▂▁▃▄▁▁▂▁▂▁▁▄▂▁▃▂█▂
eval/samples_per_second,██▅▇█▅▅██▇▇▆█▇▅▇▇▆▇▁▇
eval/steps_per_second,██▅▇█▅▅██▇▇▆█▇▅▇▇▆▇▁▇
test/accuracy,▁
test/f1_EOS,▁
+11,...


In [52]:
# 6. Save artefacts
model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)

('/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 1/models/tokenizer_config.json',
 '/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 1/models/special_tokens_map.json',
 '/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 1/models/vocab.txt',
 '/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 1/models/added_tokens.json',
 '/content/drive/MyDrive/AIRO/MNLP/homework 2/Task 1 - model 1/models/tokenizer.json')